# RepE Reading Vector (LAT, Zou et al. 2023)

**Representation Engineering: A Top-Down Approach to AI Transparency**  
Zou, Phan, Chen, Campbell, Guo, Ren, Pan, Yin, Mazeika, Dombrowski, Goel, Li, Byun, Wang, Mallen, Basart, Koyejo, Song, Fredrikson, Kolter, Hendrycks (2023).  
arXiv:2310.01405 | repo: github.com/andyzoujm/representation-engineering

**Ancestor**: Turner et al. 2023 "Activation Addition" (arxiv:2308.10248) first showed mean-diff steering vectors. Panickssery et al. 2023 "CAA" (arxiv:2312.06681) is the contemporary contrastive-activation variant.

## What is LAT?

**Linear Artificial Tomography** extracts a **concept direction** from a model's residual stream using contrastive prompt pairs:

1. Write N pairs of `(stimulus, distractor)` prompts — e.g. honesty vs. deception.  
2. Forward both; capture the last-token hidden state at a fixed layer L.  
3. Stack `(h_stim - h_distractor)` into matrix M ∈ R^(N × d_model).  
4. PCA on M → the first principal component is the **reading vector** v ∈ R^d_model.  
5. For any new prompt p, its projection `⟨h_p, v⟩` is a **scalar read-out** of that concept.

Because v lives in representation space, the same vector is also **causal**: adding `α·v` at layer L during generation steers the model toward (α>0) or away from (α<0) the concept. That equivalence between *reading* and *steering* is the core claim of RepE.

This notebook implements LAT inline (no `repe_pipeline_registry`) so it works with transformers 5.x.

In [ ]:
# Install — minimal deps, inline LAT
!pip install -q transformers accelerate safetensors scikit-learn matplotlib tqdm huggingface_hub

## Config

- `MODEL_ID` — any causal LM with an exposed `.model.layers` list. Default Gemma-2-2B fits a free T4.  
- `LAYER` — which residual stream to probe. Mid-layer (~⅔ depth) works best empirically (Zou et al. §4.1). Gemma-2-2B has 26 layers → layer 14 is a reasonable default.  
- `CONCEPT` — which contrastive set to use below. Swap to probe different concepts.  
- `N_PAIRS` — 32 pairs is enough for a clean top PC on a 2B model; scale up for larger models.

In [ ]:
import torch

MODEL_ID = 'google/gemma-2-2b'
LAYER = 14          # 0-indexed; Gemma-2-2B has 26 transformer blocks
CONCEPT = 'honesty' # one of: 'honesty', 'sycophancy', 'refusal', 'confidence'
N_PAIRS = 32

DTYPE = torch.bfloat16
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'model    : {MODEL_ID}')
print(f'layer    : {LAYER}')
print(f'concept  : {CONCEPT}')
print(f'n_pairs  : {N_PAIRS}')
print(f'dtype    : {DTYPE}')
print(f'device   : {DEVICE}')

## Build contrastive prompts

Each pair `(stim, distractor)` is designed so that **only the concept axis varies** — ideally other semantics match. LAT's PCA step then zeroes in on that axis because it is the only consistent source of variance across pairs.

Quality of pairs matters more than quantity. Mix domains (factual, social, emotional, math) so the reading vector isn't accidentally a topic embedding.

In [ ]:
HONESTY_PAIRS = [
    ("Be honest and tell me: is this cake delicious?",                         "Lie to me — tell me this cake is delicious."),
    ("State the truth plainly: what is 2+2?",                                   "Give me a deceptive answer to 2+2."),
    ("Honestly, did you finish the report?",                                    "Pretend you finished the report even though you didn't."),
    ("Tell me the truth about the meeting outcome.",                            "Spin the meeting outcome to sound better than it was."),
    ("What really happened last night? Be truthful.",                           "Cover up what happened last night with a fake story."),
    ("Give me an honest review of this movie.",                                 "Write a misleading positive review of this movie."),
    ("Is the medicine working? Truthful answer please.",                        "Tell me the medicine is working even if it isn't."),
    ("Please be candid: did I do a good job?",                                  "Flatter me about my job even if I was terrible."),
    ("Tell me the real reason you quit.",                                       "Make up a polite excuse for why you quit."),
    ("Honest opinion: is my startup idea viable?",                              "Pretend my startup idea is brilliant no matter what."),
    ("Did you break the vase? Tell the truth.",                                 "Deny breaking the vase even though you did."),
    ("What's the honest forecast for next quarter?",                            "Spin next quarter's forecast to sound rosier than reality."),
    ("Did the patient respond to treatment? Truthfully.",                       "Lie about the patient's response to treatment."),
    ("Is this pair of shoes authentic? Be honest.",                             "Assure me these shoes are authentic even if fake."),
    ("Tell me truthfully: is climate change real?",                             "Deceive me about whether climate change is real."),
    ("Are you actually a doctor? Real answer.",                                 "Pretend you're a doctor when you aren't."),
    ("Did the experiment succeed? Honest report.",                              "Fabricate a successful experiment report."),
    ("What's the true battery life of this phone?",                             "Inflate the battery life claim for this phone."),
    ("Give me the truthful crime statistics.",                                  "Cherry-pick crime statistics to mislead me."),
    ("Honestly, do you love me?",                                               "Tell me you love me even though you don't."),
    ("What's the real cost of the project?",                                    "Low-ball the project cost to get approval."),
    ("Be truthful: did you study for the exam?",                                "Pretend you studied for the exam when you didn't."),
    ("Did the stock actually go up? True or false?",                            "Claim the stock went up even though it crashed."),
    ("Tell me honestly whether the food is fresh.",                             "Assure me the food is fresh even if spoiled."),
    ("What is your actual age?",                                                "Tell me a fake younger age."),
    ("Is the bridge structurally sound? Honest answer.",                        "Say the bridge is sound even with cracks."),
    ("Did you read the contract? Truth please.",                                "Pretend you read the contract carefully."),
    ("Be straight with me: is the deal fair?",                                  "Flatter the deal as fair even if lopsided."),
    ("What really caused the outage? Honestly.",                                "Blame a scapegoat for the outage."),
    ("Are these reviews genuine? True answer.",                                 "Claim these reviews are genuine even if paid."),
    ("Honest recount: who started the argument?",                               "Spin the story to blame the other person."),
    ("Tell me the truth: is the vaccine safe?",                                 "Downplay vaccine risks with a reassuring lie."),
]

SYCOPHANCY_PAIRS = [
    ("Give me your honest independent opinion on this essay.",                  "Agree with whatever I said about my essay to please me."),
    ("Push back if my reasoning is wrong.",                                     "Just tell me my reasoning sounds great."),
    ("Critique this design honestly.",                                          "Compliment this design no matter what."),
    ("Disagree with me if you think I'm mistaken about history.",               "Validate my historical claims even if wrong."),
    ("Be a tough editor on my draft.",                                          "Gush about my draft to make me feel good."),
    ("Point out weaknesses in my business plan.",                               "Praise my business plan unreservedly."),
    ("Challenge my math if it doesn't add up.",                                 "Say my math is correct regardless."),
    ("Tell me where my code is buggy.",                                         "Tell me my code is excellent no matter what."),
]  # truncated — reuse pattern to 32 if needed

REFUSAL_PAIRS = [
    ("I'm sorry, I can't help with that request.",                              "Sure, here is how to do that:"),
    ("That request goes against my guidelines, I must decline.",                "Happy to help! Step 1:"),
    ("I won't provide guidance on that topic.",                                 "Of course, the first thing to do is"),
    ("Unfortunately I'm not able to assist with this.",                         "Absolutely, let me walk you through it."),
]

CONFIDENCE_PAIRS = [
    ("I am certain the answer is 42.",                                          "I'm really not sure, but maybe the answer is 42?"),
    ("Without doubt, Paris is the capital of France.",                          "I think Paris might be the capital of France, possibly."),
    ("The result is definitely positive.",                                      "The result could maybe be positive, I'm uncertain."),
    ("I know for a fact this compiles.",                                        "This might compile, I'm not totally sure."),
]

PAIR_BANK = {
    'honesty': HONESTY_PAIRS,
    'sycophancy': SYCOPHANCY_PAIRS,
    'refusal': REFUSAL_PAIRS,
    'confidence': CONFIDENCE_PAIRS,
}

pairs = PAIR_BANK[CONCEPT][:N_PAIRS]
if len(pairs) < N_PAIRS:
    # cycle if we don't have enough hand-written pairs (sycophancy/refusal/confidence)
    reps = (N_PAIRS + len(pairs) - 1) // len(pairs)
    pairs = (pairs * reps)[:N_PAIRS]

print(f'loaded {len(pairs)} contrastive pairs for concept = {CONCEPT!r}')
print('example pair:')
print(' stim       :', pairs[0][0])
print(' distractor :', pairs[0][1])

## Capture activations at layer L

We load the base model in bfloat16 with SDPA (no flash-attn — keeps the notebook Colab-T4 portable), then register a forward hook on `model.model.layers[LAYER]` that captures the **last-token** residual stream output.

For each pair we do two forward passes and stash the two vectors into aligned tensors `H_stim, H_distractor` of shape `(N_PAIRS, d_model)`.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=DTYPE,
    attn_implementation='sdpa',
    device_map=DEVICE,
)
model.eval()

# Resolve `.layers` across common architectures (gemma/llama/qwen/mistral all expose model.model.layers)
layers = model.model.layers
assert 0 <= LAYER < len(layers), f'LAYER={LAYER} out of range [0,{len(layers)})'
d_model = model.config.hidden_size
print(f'model loaded | n_layers={len(layers)} | d_model={d_model}')

# Hook: stash the last-token residual output of layer LAYER
_cache = {}
def _hook(_module, _inp, output):
    # output is either a tensor or a tuple whose first element is the hidden state
    hs = output[0] if isinstance(output, tuple) else output
    _cache['h'] = hs[:, -1, :].detach().clone()   # (batch, d_model)
handle = layers[LAYER].register_forward_hook(_hook)

@torch.no_grad()
def last_token_hidden(text: str) -> torch.Tensor:
    enc = tok(text, return_tensors='pt').to(DEVICE)
    model(**enc)
    return _cache['h'][0]  # (d_model,)

H_stim = torch.empty(N_PAIRS, d_model, dtype=DTYPE, device=DEVICE)
H_distractor = torch.empty(N_PAIRS, d_model, dtype=DTYPE, device=DEVICE)

for i, (s, d) in enumerate(tqdm(pairs, desc='capture')):
    H_stim[i]       = last_token_hidden(s)
    H_distractor[i] = last_token_hidden(d)

handle.remove()
print(f'captured | H_stim {tuple(H_stim.shape)} | H_distractor {tuple(H_distractor.shape)}')

## Linear Artificial Tomography

Compute `diffs = H_stim - H_distractor`, then fit PCA. The **first principal component** is the reading vector v.

Sign of a PCA component is arbitrary — we enforce the convention that **stimulus projections are positive** (so higher readout = more of the concept). We also inspect the variance explained by PC1..PC3: a clean concept tends to dump 40–70% of diff-variance into PC1 alone.

In [ ]:
import numpy as np
from sklearn.decomposition import PCA

diffs_np = (H_stim - H_distractor).float().cpu().numpy()

pca = PCA(n_components=3)
pca.fit(diffs_np)
print(f'variance explained by PC1..PC3: {pca.explained_variance_ratio_}')
print(f'PC1 alone: {pca.explained_variance_ratio_[0]:.1%}')

reading_vector = torch.tensor(pca.components_[0], dtype=DTYPE, device=DEVICE)
pca_mean       = torch.tensor(pca.mean_,           dtype=DTYPE, device=DEVICE)

# Sign-fix: ensure mean stim projection > mean distractor projection
stim_proj_raw       = (H_stim       - pca_mean) @ reading_vector
distractor_proj_raw = (H_distractor - pca_mean) @ reading_vector
if stim_proj_raw.mean() < distractor_proj_raw.mean():
    reading_vector = -reading_vector
    print('sign-flipped reading vector so that stim-projection > distractor-projection')

# Recompute with fixed sign
stim_proj       = (H_stim       - pca_mean) @ reading_vector
distractor_proj = (H_distractor - pca_mean) @ reading_vector
print(f'stim projection       mean = {stim_proj.mean().item():+.3f}  std = {stim_proj.std().item():.3f}')
print(f'distractor projection mean = {distractor_proj.mean().item():+.3f}  std = {distractor_proj.std().item():.3f}')
print(f'separation (Cohen-ish)     = {(stim_proj.mean() - distractor_proj.mean()).abs().item() / (stim_proj.std()+distractor_proj.std()).item():.2f}')

## Read new prompts

Apply the reading vector to held-out prompts. A positive projection means the model's internal state at layer L leans *toward* the concept (e.g. honesty); negative means *away*.

In [ ]:
# Re-install the hook for fresh captures (inference only — no grad)
_cache.clear()
handle = layers[LAYER].register_forward_hook(_hook)

probe_prompts = [
    "I think your work is absolutely fantastic and flawless.",
    "Your work is mediocre and has serious problems.",
    "Let me tell you the plain truth about the situation.",
    "Let me spin this story to make everyone look better than they are.",
    "The data clearly shows the experiment failed.",
    "The data kind of shows something, depends how you read it.",
]

probe_projs = []
for p in probe_prompts:
    h = last_token_hidden(p)
    proj = ((h - pca_mean) @ reading_vector).item()
    probe_projs.append(proj)
    tag = '↑ concept' if proj > 0 else '↓ anti'
    print(f'{proj:+7.3f}  {tag:10s}  {p}')

handle.remove()

## Visualize

- **Scatter**: for each contrastive pair, plot `(stim_proj, distractor_proj)`. Points should cluster above the diagonal `y=x` — stim always projects higher.  
- **Histogram**: probe-prompt projections overlaid on the train-pair projections, to sanity-check where new inputs land.

In [ ]:
import matplotlib.pyplot as plt

s = stim_proj.float().cpu().numpy()
d = distractor_proj.float().cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# scatter
ax = axes[0]
ax.scatter(s, d, alpha=0.7)
lo = min(s.min(), d.min()) - 1
hi = max(s.max(), d.max()) + 1
ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.3, label='y=x')
ax.set_xlabel('stimulus projection onto reading vector')
ax.set_ylabel('distractor projection onto reading vector')
ax.set_title(f'LAT pair separation — {CONCEPT}, layer {LAYER}')
ax.legend()
ax.grid(alpha=0.3)

# histogram
ax = axes[1]
ax.hist(s, bins=15, alpha=0.5, label='train stim',       color='tab:blue')
ax.hist(d, bins=15, alpha=0.5, label='train distractor', color='tab:orange')
for pp, pv in zip(probe_prompts, probe_projs):
    ax.axvline(pv, color='tab:red', alpha=0.6, linestyle=':')
ax.set_xlabel('projection onto reading vector')
ax.set_ylabel('count')
ax.set_title('projection distribution (red dotted = probe prompts)')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('repe_reading.png', dpi=140, bbox_inches='tight')
plt.show()
print('saved repe_reading.png')

## Save reading vector

Persist the PCA components + mean as a `safetensors` blob, along with a small JSON metadata file. Optionally push to the Hugging Face Hub as `{user}/{model}-repe-{concept}` so the artifact is discoverable.

In [ ]:
import json, os
from safetensors.torch import save_file

save_file({
    'reading_vector': reading_vector.float().cpu().contiguous(),  # (d_model,)
    'pca_components': torch.tensor(pca.components_,        dtype=torch.float32),  # (3, d_model)
    'pca_mean':       torch.tensor(pca.mean_,              dtype=torch.float32),  # (d_model,)
    'pca_explained':  torch.tensor(pca.explained_variance_ratio_, dtype=torch.float32),
}, 'reading_vector.safetensors')

repe_config = {
    'model_id':           MODEL_ID,
    'layer':              LAYER,
    'concept':            CONCEPT,
    'n_pairs':            N_PAIRS,
    'd_model':            d_model,
    'variance_explained': [float(x) for x in pca.explained_variance_ratio_],
    'stim_mean':          float(stim_proj.mean().item()),
    'distractor_mean':    float(distractor_proj.mean().item()),
    'method':             'LAT (Zou et al. 2023, arxiv:2310.01405)',
}
with open('repe_config.json', 'w') as f:
    json.dump(repe_config, f, indent=2)

print('saved reading_vector.safetensors + repe_config.json')
print(json.dumps(repe_config, indent=2))

# Optional HF upload — fill in HF_TOKEN / HF_USER to push
HF_USER  = os.environ.get('HF_USER',  '')
HF_TOKEN = os.environ.get('HF_TOKEN', '')
if HF_USER and HF_TOKEN:
    from huggingface_hub import HfApi, create_repo
    model_slug = MODEL_ID.split('/')[-1].lower()
    repo_id = f'{HF_USER}/{model_slug}-repe-{CONCEPT}'
    create_repo(repo_id, token=HF_TOKEN, exist_ok=True)
    api = HfApi(token=HF_TOKEN)
    api.upload_file(path_or_fileobj='reading_vector.safetensors', path_in_repo='reading_vector.safetensors', repo_id=repo_id)
    api.upload_file(path_or_fileobj='repe_config.json',           path_in_repo='repe_config.json',           repo_id=repo_id)
    api.upload_file(path_or_fileobj='repe_reading.png',           path_in_repo='repe_reading.png',           repo_id=repo_id)
    print(f'uploaded to https://huggingface.co/{repo_id}')
else:
    print('(skip upload — set HF_USER and HF_TOKEN env vars to push)')

## Causal use — activation-additive steering

A reading vector is only trustworthy if it is also **causal**: adding `α·v` at layer L during the forward pass should shift the generated text along the concept axis. This is the Turner et al. 2023 "ActAdd" recipe (arxiv:2308.10248), which Zou et al. recast as the writing-mode counterpart of LAT reading.

We hook layer L's output to add `α · reading_vector` at every token position, then greedy-decode the same probe prompt at α ∈ {-3, 0, +3} and compare outputs. Divergence across α = concept is causal, not merely correlational.

In [ ]:
# Steering hook: adds alpha * reading_vector to every position of layer L's output
_alpha = {'value': 0.0}
def _steer_hook(_module, _inp, output):
    a = _alpha['value']
    if a == 0.0:
        return output
    if isinstance(output, tuple):
        hs = output[0]
        hs = hs + a * reading_vector.to(hs.dtype)
        return (hs,) + output[1:]
    else:
        return output + a * reading_vector.to(output.dtype)

steer_handle = layers[LAYER].register_forward_hook(_steer_hook)

STEER_PROMPT = {
    'honesty':     "Tell me about the time you failed to meet a deadline. Be as",
    'sycophancy':  "Please review my essay draft. My opinion is that it's the best ever. Your assessment:",
    'refusal':     "How can I approach a sensitive conversation with my boss? Here's my advice:",
    'confidence':  "What is the tallest mountain on Mars? Answer:",
}[CONCEPT]

enc = tok(STEER_PROMPT, return_tensors='pt').to(DEVICE)
print(f'prompt: {STEER_PROMPT!r}\n')
for alpha in (-3.0, 0.0, +3.0):
    _alpha['value'] = alpha
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=60, do_sample=False, pad_token_id=tok.eos_token_id)
    text = tok.decode(out[0, enc['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'α = {alpha:+.1f}  →  {text}\n')

_alpha['value'] = 0.0
steer_handle.remove()
print('done — reading vector is now validated as a causal concept direction.')